# Week 8 (Notebook 2): Encoder‑Decoder Generation (T5)

Goal: learn **seq2seq** (encoder‑decoder) generation via the **text‑to‑text** framing popularized by T5.

We’ll do:
- a small translation task (English → German)
- CPU‑friendly fine‑tuning defaults (small slices + few steps)

Notes:
- First run will download a dataset/model from Hugging Face.
- Keep dataset sizes and steps small for CPU.

In [ ]:
# Setup
import os
import random
import numpy as np
import torch

seed = 204
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
device

## Part A — What changes vs GPT?

**Decoder‑only (GPT):** one stack, causal self‑attention, predicts next token.

**Encoder‑decoder (T5):**
- **Encoder** reads the full source sequence with bidirectional self‑attention.
- **Decoder** generates the target with causal self‑attention **and** cross‑attention into the encoder states.
- Training uses teacher forcing: decoder sees the gold prefix, learns to predict the next target token.

## Part B — Hugging Face T5 translation (CPU‑friendly)

We use the dataset from the course roadmap: `opus100` (en‑de).

To keep this realistic on CPU we:
- take small slices
- cap max source/target length
- train for a small number of steps

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

hf_model_name = "t5-small"

# CPU-friendly sizes
n_train = 4000
n_val = 500

max_source_len = 128
max_target_len = 128

raw = load_dataset("opus100", "en-de")
raw

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(hf_model_name)

prefix = "translate English to German: "

def preprocess(batch):
    src_texts = [prefix + ex["en"] for ex in batch["translation"]]
    tgt_texts = [ex["de"] for ex in batch["translation"]]

    model_inputs = tokenizer(
        src_texts,
        max_length=max_source_len,
        truncation=True,
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            tgt_texts,
            max_length=max_target_len,
            truncation=True,
        )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tok = raw.map(preprocess, batched=True, remove_columns=raw["train"].column_names)

train_ds = tok["train"].shuffle(seed=seed).select(range(min(n_train, len(tok["train"])) ))
val_ds = tok["validation"].shuffle(seed=seed).select(range(min(n_val, len(tok["validation"])) ))
train_ds, val_ds

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(hf_model_name).to(device)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

out_dir = "./models/week8_t5small_opus100_en_de"
os.makedirs(out_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=out_dir,
    overwrite_output_dir=True,
    max_steps=300,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-4,
    weight_decay=0.01,
    warmup_steps=30,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    report_to="none",
    seed=seed,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer

In [ ]:
# CPU-realistic run: keep this small
# trainer.train()

# If you already trained, point to a checkpoint folder here:
# ckpt = "./models/week8_t5small_opus100_en_de/checkpoint-300"
# model = AutoModelForSeq2SeqLM.from_pretrained(ckpt).to(device)

print("Ready: uncomment trainer.train() to fine-tune.")

In [ ]:
# Translation demo (works before/after fine-tuning)
from transformers import set_seed

set_seed(seed)
src = "I really enjoyed this movie, but the ending was disappointing."
inp = tokenizer(prefix + src, return_tensors="pt").to(device)

gen = model.generate(
    **inp,
    max_new_tokens=80,
    num_beams=4,
)

print("EN:", src)
print("DE:", tokenizer.decode(gen[0], skip_special_tokens=True))